In [ ]:
# %pip install plotly


In [ ]:
import cppyy
import os
import numpy as np

import planner_plot
from planner_plot import plot_trapezoid

# Choose trajectory generator: "trapezoidal" or "poly5"
TRAJ_TYPE = "trapezoidal"
# TRAJ_TYPE = "poly5"


def load_miniplanner_and_traj():
    # Mirror the ISR notebook style: include headers/sources on-demand.
    root = os.path.abspath(".")
    if not hasattr(cppyy.gbl, "mp_reset"):
        cppyy.add_include_path(root)
        cppyy.add_include_path(os.path.join(root, "inc"))
        cppyy.add_include_path(os.path.join(root, "marlin files"))
        cppyy.include("miniplanner.h")
        # Implementation lives in a .cpp, so we include it (JIT-compiled by cppyy/cling).
        cppyy.include("miniplanner.cpp")

    if not hasattr(cppyy.gbl, "TrapezoidalTrajectoryGenerator"):
        cppyy.add_include_path(root)
        cppyy.add_include_path(os.path.join(root, "inc"))
        cppyy.add_include_path(os.path.join(root, "marlin files"))
        cppyy.include("marlin files/trajectory_generator.h")
        cppyy.include("marlin files/trajectory_trapezoidal.h")
        cppyy.include("marlin files/trajectory_poly5.h")


load_miniplanner_and_traj()


In [ ]:
# Convenience aliases matching the old ctypes-based API
mp_reset = cppyy.gbl.mp_reset
mp_push_block = cppyy.gbl.mp_push_block
mp_recalculate = cppyy.gbl.mp_recalculate
mp_size = cppyy.gbl.mp_size
mp_get_block = cppyy.gbl.mp_get_block
mp_pop_front = cppyy.gbl.mp_pop_front

MP_BlockOut = cppyy.gbl.MP_BlockOut


def get_block(i: int):
    out = MP_BlockOut()
    ok = mp_get_block(i, out)
    assert int(ok) == 1
    return {
        "mm": float(out.millimeters),
        "nom": float(out.nominal_speed),
        "accel": float(out.acceleration),
        "entry": float(out.entry_speed),
        "exit": float(out.exit_speed),
        "max_entry": float(out.max_entry_speed),
        "min_entry": float(out.min_entry_speed),
    }


def make_traj_generator(traj_type: str):
    traj_type = traj_type.lower()
    if traj_type == "trapezoidal":
        return cppyy.gbl.TrapezoidalTrajectoryGenerator()
    if traj_type == "poly5":
        return cppyy.gbl.Poly5TrajectoryGenerator()
    raise ValueError(f"Unknown TRAJ_TYPE: {traj_type}")


def run_trapezoid_profile(blocks, dt=0.0005, traj_type=TRAJ_TYPE):
    traj = make_traj_generator(traj_type)

    times = []
    positions = []
    boundaries = []

    t_offset = 0.0
    x_offset = 0.0

    for b in blocks:
        traj.reset()
        traj.plan(
            float(b["entry"]),
            float(b["exit"]),
            float(b["accel"]),
            float(b["nom"]),
            float(b["mm"]),
        )

        duration = float(traj.getTotalDuration())
        if duration <= 0.0:
            boundaries.append(t_offset)
            continue

        steps = max(2, int(np.ceil(duration / dt)))
        local_ts = np.linspace(0.0, duration, steps)
        local_xs = [float(traj.getDistanceAtTime(float(t))) for t in local_ts]

        times.extend((t_offset + t) for t in local_ts)
        positions.extend((x_offset + x) for x in local_xs)

        t_offset += duration
        x_offset += float(b["mm"])
        boundaries.append(t_offset)

    return np.array(times), np.array(positions), np.array(boundaries)


print("loaded OK via cppyy")


In [ ]:
mp_reset()

# Add a few blocks: (mm, nominal mm/s, accel mm/s^2, max_entry mm/s, min_entry_speed mm/s)
blocks = [
    (35.0, 200.0, 500.0, 10, 10),
    (35.0, 100.0, 500.0, 10, 0),
    (35.0, 100.0, 500.0, 30, 0),
    (35.0, 100.0, 500.0, 40, 0),
    (35.0, 100.0, 500.0, 50, 0),
]
total_dist_planned = 0
for mm, nominal, accel, max_entry, min_entry_speed in blocks:
    total_dist_planned += mm
    ok = mp_push_block(mm, nominal, accel, max_entry, min_entry_speed)
    assert int(ok) == 1

# For a standalone queue, safe exit is typically 0 (come to a stop).
mp_recalculate(0.0)

n = int(mp_size())
print("n blocks:", n)
blocks_out = [get_block(i) for i in range(n)]
for i, b in enumerate(blocks_out):
    print(i, b)

# Run trapezoidal trajectory per block and plot position/velocity

t, x, boundaries = run_trapezoid_profile(blocks_out)
if len(t) > 1:
    v = np.gradient(x, t)
    plot_trapezoid(t, x, v, boundaries_s=boundaries)

print("total_dist_planned:", total_dist_planned)
print("executed:", x[-1])